#  Model Evaluation Framework: Integrating EDA and Business Strategy

Our evaluation framework goes beyond mere technical performance measurement; it serves as a bridge between the empirical reality discovered during the **Exploratory Data Analysis (EDA)** and the pragmatic need to optimize vehicle portfolio management and resale operations.

###  Model Selection: Linking Theory to EDA
We have selected three models representing different levels of complexity and learning capabilities: **Linear Regression**, used as a Baseline; **Random Forest**, acting as the Guardian of Stability; and **XGBoost**, defined as the Margin Optimizer.

### In-Depth Analysis of Evaluation Metrics
The choice of metrics is driven by our corporate risk assessment. **MAPE (Mean Absolute Percentage Error)** represents our "business survival metric." Unlike other measures, it transforms the error from an absolute monetary value into a percentage relative to the actual sale price. This distinction is vital for several reasons emerged during our study, starting with consistency with margin structures: in the used car market, a dealer's profit margin is typically calculated as a fixed percentage of the vehicle's value (usually between 10% and 15%).

From an operational risk perspective, if the model shows a MAPE of 15% against a commercial margin of 10%, the algorithm would suggest prices that could push the company into systemic losses on every single transaction, nullifying the expected gain. Furthermore, this metric is crucial for managing the portfolio heterogeneity revealed by EDA insights, which showed a wide price variability ranging from £2,000 to £50,000. For the business, a £1,000 error on a £4,000 city car is an operational failure with a totally destroyed margin and an out-of-market price, whereas the same error on a £45,000 luxury car is considered negligible. MAPE normalizes the error, ensuring the model is accurate proportionally to the asset's value, regardless of the segment. Finally, MAPE acts as a counter-measure to skewed distribution: since the EDA highlighted an unbalanced distribution with many budget cars and few luxury ones, using MAPE prevents the model from falling into "statistical laziness." Without this metric, the algorithm might tend to optimize only the prices of expensive cars to minimize absolute error (MAE/RMSE), neglecting budget vehicles; MAPE, instead, forces the model to maintain high precision across the entire catalog, protecting mass-market sales volumes.



$$MAPE = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{\text{Actual}_i - \text{Predicted}_i}{\text{Actual}_i} \right| \times 100$$

Parallelly, **RMSE (Root Mean Squared Error)** represents our financial risk management system. This metric does not just measure the average error but acts as a critical sensor for the stability of the dealership's working capital. The distinctive mathematical feature of RMSE is the squaring of residuals before averaging. This means errors are not summed linearly: a £5,000 error is not considered five times worse than a £1,000 one, but twenty-five times more severe in terms of penalty on the cost function. For a dealer, small, widespread errors are operationally manageable, but a single "miss" on a high-value car can jeopardize an entire month's profitability. RMSE forces the model to be extremely cautious, minimizing deviations that could lead to catastrophic losses.

During the EDA phase, we identified "Premium" vehicle segments with prices up to £50,000, representing significant capital investment. A massive valuation error on these units—for example, overestimating a high-end Mercedes or Audi—would lead to an out-of-market price, resulting in "dead stock." Financially, this translates into capital "stuck" in the warehouse: the inability to quickly sell a £50k asset deprives the dealership of the liquidity needed to acquire new, more dynamic stock, reducing inventory turnover. Since RMSE is highly sensitive to outliers, it allows us to monitor if the model is failing on specific car models or sub-segments identified in the EDA. If RMSE remains high despite a good MAPE, we know the model is generally accurate but tends to make sporadic yet destructive errors on critical vehicles. We use RMSE to ensure our ensemble models (Random Forest and XGBoost) are effectively regularizing predictions and not producing "wild" prices that the dealership's financial structure could not absorb.



$$RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\text{Actual}_i - \text{Predicted}_i)^2}$$

**Adjusted $R^2$** evaluates the scientific quality of our model's structure, representing the fundamental parameter to validate our feature engineering process and protect the system from overfitting. By nature, the standard coefficient of determination ($R^2$) is a "greedy" metric: its value increases or stays the same every time we add a new variable, regardless of whether it is truly useful for predicting price. If we relied on simple $R^2$, we would be tempted to include infinite categories, such as every single color variant or minor optional, just to see the score rise, ending up modeling statistical "noise" rather than the actual market signal.

Adjusted $R^2$ introduces a mathematical correction based on the ratio between the number of observations ($n$) and the number of predictors ($k$); if we add a variable that does not bring a significant improvement to predictive power, the Adjusted $R^2$ will decrease. This metric thus acts as an impartial referee, telling us if the complexity introduced by variables like Brand, Transmission, or Fuel Type is justified by a real increase in precision or if it is just unnecessarily burdening the model. During the data preparation phase, we transformed raw categorical variables into numerous numerical columns via One-Hot Encoding. For a car dealer, an overly complex model is hard to maintain and risks giving too much importance to irrelevant features. Adjusted $R^2$ ensures the final model is as lean and effective as possible (model parsimony), guaranteeing that our portfolio segmentation is statistically sound and that the model will generalize well to new cars entering inventory in the future, avoiding predictions biased by coincidences present only in the historical dataset.



$$R_{adj}^{2} = 1 - \left[ \frac{(1 - R^{2})(n - 1)}{n - k - 1} \right]$$

###  Candidate Metrics and Rationale for Exclusion
In this project, we evaluated the integration of alternative criteria based on automotive industry literature and techniques presented during the course, deciding however to exclude them for strategic and operational reasons.

**MedAE (Median Absolute Error)** calculates the median of the absolute difference between actual and predicted prices. Unlike the mean, the median is not influenced by extreme values. The used car market is historically "noisy" and presents strong anomalies due to incorrect listings, cars sold at out-of-market prices for sentimental reasons, or data entry errors. Theoretically, MedAE would be the most robust metric because it ignores these outliers, offering a view of "typical" model performance. However, although the median offers an optimistic view, for a car dealer, outliers represent the greatest financial risk. If the model misses the valuation of a high-end Mercedes by £10,000, the economic damage to the dealership is real, immediate, and heavy. Using the median would mean "sweeping under the rug" these critical failures. In a business where every vehicle is an investment of immobilized capital, we preferred metrics like MAE (mean) and RMSE (which penalizes large errors), because every single pound lost in a valuation error must be visible and accounted for.

**Quantile Loss (Quantile Regression)** is used to estimate the quantiles of the target distribution, allowing for the prediction of a confidence interval rather than a single value. We could have used it to provide the seller with a "price range" (e.g., "Minimum Liquidation Price" and "Maximum Market Price"), offering flexibility during customer negotiations. However, in the context of Inventory Planning, ambiguity is a cost. The dealer needs a unique list price to automate ad publication on online portals and to accurately calculate the total portfolio value on the balance sheet. Introducing price ranges would increase the decision-making complexity for the sales team without guaranteeing a higher economic return than the point estimation provided by optimized models like XGBoost.

Finally, **Training Time (Computational Efficiency)** measures the time required for the algorithm to complete the training phase. In Big Data contexts, computational efficiency is a critical KPI for choosing between simple and complex models. Our post-cleaning dataset contains approximately 94,000 observations, and any modern hardware processes this volume of data in a few minutes. Unlike the stock market (High-Frequency Trading), used car prices do not fluctuate every millisecond. Training speed is not a competitive factor for our Business Problem; prediction accuracy (Optimal Pricing) has absolute priority over calculation time, as the value created by correct pricing far outweighs the cost of a few extra minutes of computation.